In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install Docker (Note: Docker support in Colab is limited)
!apt-get update
!apt-get install -y docker.io

# Download OSM data for your region to your Drive
!mkdir -p /content/drive/MyDrive/osrm_data
!wget -P /content/drive/MyDrive/osrm_data http://download.geofabrik.de/south-america/colombia-latest.osm.pbf

Mounted at /content/drive
Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,931 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,245 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-

In [2]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/columbia_full_coordinates.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Harvesine distance

In [3]:
df

,date,region,Center_ID,Client_ID,ordered_boxes,Latitude_CD,Longitude_CD,Latitude_Client,Longitude_Client,ordered_volume
0,2025-03-18,Sur,AV46,11645498,50.00,3.535408,-76.492858,3.449400,-76.522300,3.780000
1,2025-03-01,Centro,AV47,12307950,12.02,4.541264,-74.251706,4.562510,-74.236074,0.908712
2,2025-03-06,Centro,AV26,13212774,2.00,4.596453,-74.148856,4.563009,-74.104803,0.151200
3,2025-03-31,Norte,AV39,12946586,102.25,8.729500,-75.847328,8.953745,-75.838556,7.730100
4,2025-03-20,Sur,AV18,13682698,6.00,2.915592,-75.285719,2.924496,-75.259561,0.453600
...,...,...,...,...,...,...,...,...,...,...
406924,2025-03-20,Centro,AV47,10640224,8.50,4.541264,-74.251706,4.615580,-74.351343,0.642600
406925,2025-03-15,Andes,AV05,14260184,5.75,6.168617,-75.618514,6.171558,-75.610056,0.434700
406926,2025-03-12,Centro,AV06,14208244,32.00,4.946467,-73.941175,4.908358,-73.942483,2.419200
406927,2025-03-18,Sur,AV24,12610824,19.52,5.280189,-72.427817,5.181973,-72.572259,1.475712


In [4]:
df_centro = df[df['region'] == 'Centro']
df_centro


,date,region,Center_ID,Client_ID,ordered_boxes,Latitude_CD,Longitude_CD,Latitude_Client,Longitude_Client,ordered_volume
1,2025-03-01,Centro,AV47,12307950,12.02,4.541264,-74.251706,4.562510,-74.236074,0.908712
2,2025-03-06,Centro,AV26,13212774,2.00,4.596453,-74.148856,4.563009,-74.104803,0.151200
5,2025-03-12,Centro,AV04,11413970,9.00,5.807142,-73.006556,5.704820,-72.971300,0.680400
10,2025-03-31,Centro,AV04,10287272,6.00,5.807142,-73.006556,5.836970,-73.023700,0.453600
11,2025-03-20,Centro,AV47,10766132,17.00,4.541264,-74.251706,4.585490,-74.171600,1.285200
...,...,...,...,...,...,...,...,...,...,...
406915,2025-03-10,Centro,AV06,13988701,12.00,4.946467,-73.941175,4.706422,-74.055773,0.907200
406918,2025-03-11,Centro,AV26,13185779,8.00,4.596453,-74.148856,4.506860,-74.116000,0.604800
406921,2025-03-31,Centro,AV42,12308514,3.50,5.544456,-73.352553,5.032237,-73.104175,0.264600
406924,2025-03-20,Centro,AV47,10640224,8.50,4.541264,-74.251706,4.615580,-74.351343,0.642600


In [5]:
# prompt: check whether there are "Client_ID" values which appear multiple times

duplicate_client_ids = df_centro[df_centro.duplicated(subset=['Client_ID'], keep=False)]

if not duplicate_client_ids.empty:
  print("Duplicate Client_IDs found:")
  print(duplicate_client_ids['Client_ID'])
else:
  print("No duplicate Client_IDs found.")


Duplicate Client_IDs found:
1         12307950
2         13212774
5         11413970
10        10287272
11        10766132
            ...   
406911    10768374
406915    13988701
406921    12308514
406924    10640224
406926    14208244
Name: Client_ID, Length: 113907, dtype: int64


In [6]:
# prompt: for each Client_ID, sum the ordered_boxes and ordered_volume. the data frame contains now every client number only once with the ordered_boxes and ordered volume value as the sum. keep the values of the other columns the same way

grouped_df = df_centro.groupby('Client_ID').agg(
    ordered_boxes=('ordered_boxes', 'sum'),
    ordered_volume=('ordered_volume', 'sum')
).reset_index()

# Merge the aggregated data back to the original DataFrame
df_centro = pd.merge(grouped_df, df_centro.drop_duplicates(subset='Client_ID').drop(['ordered_boxes', 'ordered_volume'], axis=1), on='Client_ID', how='left')
df_centro


,Client_ID,ordered_boxes,ordered_volume,date,region,Center_ID,Latitude_CD,Longitude_CD,Latitude_Client,Longitude_Client
0,10279095,8955.000000,676.998000,2025-03-10,Centro,AV04,5.807142,-73.006556,5.806170,-73.005640
1,10279105,3129.750000,236.609100,2025-03-26,Centro,AV42,5.544456,-73.352553,5.403232,-73.335933
2,10279115,8814.750000,666.395100,2025-03-28,Centro,AV04,5.807142,-73.006556,5.825424,-73.032050
3,10279118,2647.650000,200.162340,2025-03-28,Centro,AV42,5.544456,-73.352553,5.631301,-73.527817
4,10279127,41.000000,3.099600,2025-03-18,Centro,AV04,5.807142,-73.006556,5.836823,-73.041105
...,...,...,...,...,...,...,...,...,...,...
54051,14267532,188.366667,14.240520,2025-03-13,Centro,AV47,4.541264,-74.251706,4.616088,-74.351446
54052,14267595,51.208333,3.871350,2025-03-23,Centro,AV42,5.544456,-73.352553,5.518601,-73.363196
54053,14267770,9.520000,0.719712,2025-03-06,Centro,AV26,4.596453,-74.148856,4.590625,-74.103432
54054,14267787,28.000000,2.116800,2025-03-20,Centro,AV06,4.946467,-73.941175,4.868191,-74.060665


In [7]:
import numpy as np

# Assuming df_centro is your DataFrame
clients = df_centro[['Client_ID', 'Latitude_Client', 'Longitude_Client']]
centers = df_centro[['Center_ID', 'Latitude_CD', 'Longitude_CD']].drop_duplicates().reset_index(drop=True)


### Harvesine distance

In [8]:
def haversine(lat1, lon1, lat2, lon2):
    # Radius of Earth in kilometers
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2.0)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Compute distance matrix: rows=clients, cols=centers
distance_matrix = np.zeros((len(clients), len(centers)))
for i, client in clients.iterrows():
    for j, center in centers.iterrows():
        distance_matrix[i, j] = haversine(client['Latitude_Client'], client['Longitude_Client'],
                                          center['Latitude_CD'], center['Longitude_CD'])


In [9]:
!pip install ortools
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('SCIP')
num_clients = len(clients)
num_centers = len(centers)

# Decision variables: x[i, j] = 1 if client i assigned to center j
x = {}
for i in range(num_clients):
    for j in range(num_centers):
        x[i, j] = solver.BoolVar(f'x_{i}_{j}')

# Constraint: each client assigned to exactly one center
for i in range(num_clients):
    solver.Add(solver.Sum([x[i, j] for j in range(num_centers)]) == 1)

# (Optional) Center capacity constraints can be added here if needed

# Objective: minimize total distance
objective_terms = []
for i in range(num_clients):
    for j in range(num_centers):
        objective_terms.append(distance_matrix[i, j] * x[i, j])
solver.Minimize(solver.Sum(objective_terms))

# Solve
status = solver.Solve()
if status == pywraplp.Solver.OPTIMAL:
    print('Optimal assignment found.')
    # Extract assignments
    assignments = []
    for i in range(num_clients):
        for j in range(num_centers):
            if x[i, j].solution_value() > 0.5:
                assignments.append((clients.iloc[i]['Client_ID'], centers.iloc[j]['Center_ID']))
else:
    print('No optimal solution found.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 9.7 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
Optimal assignment found.


In [10]:
assignments

[(np.float64(10279095.0), 'AV04'),
 (np.float64(10279105.0), 'AV42'),
 (np.float64(10279115.0), 'AV04'),
 (np.float64(10279118.0), 'AV42'),
 (np.float64(10279127.0), 'AV04'),
 (np.float64(10279130.0), 'AV04'),
 (np.float64(10279132.0), 'AV04'),
 (np.float64(10279133.0), 'AV04'),
 (np.float64(10279142.0), 'AV04'),
 (np.float64(10279144.0), 'AV04'),
 (np.float64(10279146.0), 'AV04'),
 (np.float64(10279147.0), 'AV04'),
 (np.float64(10279149.0), 'AV04'),
 (np.float64(10279152.0), 'AV04'),
 (np.float64(10279155.0), 'AV04'),
 (np.float64(10279164.0), 'AV04'),
 (np.float64(10279165.0), 'AV04'),
 (np.float64(10279168.0), 'AV04'),
 (np.float64(10279186.0), 'AV04'),
 (np.float64(10279188.0), 'AV04'),
 (np.float64(10279190.0), 'AV04'),
 (np.float64(10279193.0), 'AV04'),
 (np.float64(10279200.0), 'AV04'),
 (np.float64(10279205.0), 'AV04'),
 (np.float64(10279206.0), 'AV04'),
 (np.float64(10279207.0), 'AV04'),
 (np.float64(10279209.0), 'AV04'),
 (np.float64(10279211.0), 'AV04'),
 (np.float64(1027921

### Solve with Clustering

In [11]:
# Cell 1: Perform Clustering
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import time

def calculate_clusters(df_centro, n_clusters=10):
    """Cluster clients into geographical groups"""
    print(f"Clustering {len(df_centro)} clients into {n_clusters} groups...")
    start_time = time.time()

    # Extract client coordinates
    client_coords = np.array(list(zip(
        df_centro['Latitude_Client'],
        df_centro['Longitude_Client']
    )))

    # Perform K-means clustering
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    ).fit(client_coords)

    # Add cluster IDs to DataFrame
    df_centro['cluster_id'] = kmeans.labels_

    # Calculate execution time
    duration = time.time() - start_time
    print(f"Clustering completed in {duration:.1f} seconds")
    print(f"Cluster sizes:\n{df_centro['cluster_id'].value_counts().describe()}")

    return df_centro

# Execute clustering
try:
    df_centro = calculate_clusters(df_centro)
except Exception as e:
    print(f"Clustering failed: {str(e)}")
    raise


Clustering 54056 clients into 10 groups...
Clustering completed in 1.2 seconds
Cluster sizes:
count       10.000000
mean      5405.600000
std       5160.669814
min        991.000000
25%       1973.750000
50%       3513.500000
75%       6586.000000
max      18031.000000
Name: count, dtype: float64


In [12]:
!pip install plotly

import plotly.express as px

# Assuming df_centro has 'Latitude_Client', 'Longitude_Client', and 'cluster_id' columns
fig = px.scatter_mapbox(df_centro,
                        lat="Latitude_Client",
                        lon="Longitude_Client",
                        color="cluster_id",
                        zoom=8,
                        height=600,
                        width=800,
                        mapbox_style="carto-positron") # or open-street-map, stamen-terrain, etc.

fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()


In [13]:
# Cell 2: Calculate Cluster Centroids
def calculate_centroids(df_centro):
    """Calculate geographic centroids for each cluster"""
    print("\nCalculating cluster centroids...")
    start_time = time.time()

    n_clusters = df_centro['cluster_id'].nunique()
    centroids = []

    for cluster_id in range(n_clusters):
        cluster_df = df_centro[df_centro['cluster_id'] == cluster_id]

        if len(cluster_df) == 0:
            raise ValueError(f"Cluster {cluster_id} is empty")

        centroid = (
            cluster_df['Latitude_Client'].mean(),
            cluster_df['Longitude_Client'].mean()
        )
        centroids.append(centroid)

    # Add centroids to a DataFrame
    centroids_df = pd.DataFrame(centroids, columns=['Latitude', 'Longitude'])
    centroids_df['cluster_id'] = range(n_clusters)

    duration = time.time() - start_time
    print(f"Calculated {len(centroids)} centroids in {duration:.1f} seconds")

    return centroids_df

# Execute centroid calculation
try:
    centroids_df = calculate_centroids(df_centro)
except Exception as e:
    print(f"Centroid calculation failed: {str(e)}")
    raise



Calculating cluster centroids...
Calculated 10 centroids in 0.1 seconds


In [14]:
# Cell 3: Assign Clusters to DCs via API
import requests
from tqdm import tqdm

def assign_clusters_to_dcs(centroids_df, df_centro):
    """Assign clusters to distribution centers using OSRM API"""
    print("\nStarting DC assignment process...")
    start_time = time.time()

    # Get DC coordinates
    centers = df_centro[['Center_ID', 'Latitude_CD', 'Longitude_CD']].drop_duplicates()
    dc_coords = list(zip(centers['Latitude_CD'], centers['Longitude_CD']))
    dc_ids = centers['Center_ID'].tolist()

    # Storage for assignments
    cluster_assignments = {}
    distance_matrix = []

    try:
        # Process each cluster with progress bar
        for idx, row in tqdm(centroids_df.iterrows(), total=len(centroids_df), desc="API Progress"):
            centroid = (row['Latitude'], row['Longitude'])
            distances = []

            for dc in dc_coords:
                url = f"https://router.project-osrm.org/route/v1/driving/" \
                      f"{centroid[1]},{centroid[0]};{dc[1]},{dc[0]}?overview=false"

                response = requests.get(url, timeout=60)
                response.raise_for_status()

                data = response.json()
                distances.append(data['routes'][0]['distance'] / 1000)  # Convert to km

                # Rate limiting
                time.sleep(0.1)

            # Find closest DC
            closest_dc_idx = np.argmin(distances)
            cluster_assignments[row['cluster_id']] = dc_ids[closest_dc_idx]
            distance_matrix.append(distances)

    except Exception as e:
        print(f"\nAPI call failed at cluster {row['cluster_id']}")
        print(f"Error: {str(e)}")
        raise

    # Assign final DCs to clients
    df_centro['Assigned_CD'] = df_centro['cluster_id'].map(cluster_assignments)

    # Add results to centroids DataFrame
    centroids_df['Assigned_DC'] = centroids_df['cluster_id'].map(cluster_assignments)
    centroids_df['Distances'] = distance_matrix

    duration = time.time() - start_time
    print(f"\nAssignment completed in {duration:.1f} seconds")
    print("Sample assignments:")
    print(centroids_df[['cluster_id', 'Assigned_DC']].head())

    return df_centro, centroids_df

# Execute assignment
try:
    df_centro, centroids_df = assign_clusters_to_dcs(centroids_df, df_centro)
except Exception as e:
    print(f"Assignment failed: {str(e)}")
    raise



Starting DC assignment process...


API Progress:   0%|          | 0/10 [01:00<?, ?it/s]


API call failed at cluster 0.0
Error: HTTPSConnectionPool(host='router.project-osrm.org', port=443): Read timed out. (read timeout=60)
Assignment failed: HTTPSConnectionPool(host='router.project-osrm.org', port=443): Read timed out. (read timeout=60)


ReadTimeout: HTTPSConnectionPool(host='router.project-osrm.org', port=443): Read timed out. (read timeout=60)

In [ ]:
# Display the clients and their assigned distribution centers
print(df_centro[['Client_ID', 'Assigned_CD']])


In [ ]:
# Check for differences in assignment between 'Assigned_CD' and 'Center_ID'
differences = df_centro[df_centro['Assigned_CD'] != df_centro['Center_ID']]

if not differences.empty:
    print("Clients with different assignments:")
    print(differences[['Client_ID', 'Center_ID', 'Assigned_CD']])
else:
    print("No clients have different assignments between 'Assigned_CD' and 'Center_ID'.")
